In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
# ติดตั้ง rpy2 (เชื่อมระหว่าง Python และ R)
!pip install rpy2

# โหลด extension เพื่อใช้ R ใน Jupyter/Colab
%load_ext rpy2.ipython


In [ ]:
%%R
# ติดตั้งแพ็กเกจ (รันแค่ครั้งเดียว)
install.packages("INLA", repos=c(getOption("repos"),
        INLA="https://inla.r-inla-download.org/R/stable"), dep=TRUE)
install.packages("spdep")
install.packages("sf")
install.packages("tidyverse")
install.packages("readxl")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘gsl’, ‘runjags’

trying URL 'https://cran.rstudio.com/src/contrib/gsl_2.1-8.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/runjags_2.2.2-5.tar.gz'
trying URL 'https://inla.r-inla-download.org/R/stable/src/contrib/INLA_24.12.11.tar.gz'

The downloaded source packages are in
	‘/tmp/RtmpBFeRgL/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/spdep_1.3-11.tar.gz'
Content type 'application/x-gzip' length 4685359 bytes (4.5 MB)
downloaded 4.5 MB


The downloaded source packages are in
	‘/tmp/RtmpBFeRgL/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/sf_1.0-20.tar.gz'
Content type 'application/x-gzip' length 4492197 bytes (4.3 MB)
downloaded 4.3 MB


The downloaded 

In [ ]:
%%R
library(INLA)
library(spdep)
library(sf)
library(tidyverse)
library(readxl)


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ tidyr::expand() masks Matrix::expand()
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
✖ tidyr::pack()   masks Matrix::pack()
✖ tidyr::unpack() masks Matrix::unpack()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loading required package: Matrix
This is INLA_24.12.11 built 2024-12-11 19:58:26 UTC.
 - See www.r-inla.org/contact-us for how to get help.
 - List available models/likelihoods/etc with inla.list.models()
 - Use inla.doc(<NAME>) to access documentation
Loading required package: spData
To access larger datasets in this package, install the spDataLarge
package with: `install.packages('spDataLarge',
repos='https://nowosad.github.io/drat/', type='source')`
Loading required package: sf
Linking to GEOS 3.11.1, GDAL 3.6.4, PROJ 9.1.1; sf_use_s2() is TRUE


In [ ]:
%%R
# อ่านข้อมูล
data <- read_excel("/content/gdrive/MyDrive/Senior Project/Spatial/DATA/Klebsiella pneumoniae/Ertapenem/PERCENT_R_Klebsiella pneumoniae_Ertapenem.xlsx")
data

# A tibble: 1,042 × 6
   GROUP_NAME            ANTIBIOTIC REGION SPEC_DATE  Year R_percent
   <chr>                 <chr>       <dbl> <chr>     <dbl>     <dbl>
 1 Klebsiella pneumoniae Ertapenem       1 2015-01    2015      1.75
 2 Klebsiella pneumoniae Ertapenem       7 2015-01    2015      4.08
 3 Klebsiella pneumoniae Ertapenem      13 2015-01    2015     10   
 4 Klebsiella pneumoniae Ertapenem       1 2015-02    2015      1.98
 5 Klebsiella pneumoniae Ertapenem       7 2015-02    2015      4.44
 6 Klebsiella pneumoniae Ertapenem      13 2015-02    2015      4.17
 7 Klebsiella pneumoniae Ertapenem       1 2015-03    2015      0   
 8 Klebsiella pneumoniae Ertapenem       7 2015-03    2015      2.22
 9 Klebsiella pneumoniae Ertapenem      13 2015-03    2015      2.94
10 Klebsiella pneumoniae Ertapenem       1 2015-04    2015      3.2 
# ℹ 1,032 more rows
# ℹ Use `print(n = ...)` to see more rows


In [ ]:
%%R
# เข้ารหัสพื้นที่ (province) และเวลา (month)
data$Region_id <- as.numeric(as.factor(data$REGION))
# สร้าง factor แล้วแปลงเป็นเลขที่เรียง
data$month_id <- as.numeric(as.factor(data$SPEC_DATE))

data

# A tibble: 1,042 × 8
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <dbl>    <dbl>
 1 Klebsiella pn… Ertapenem       1 2015-01    2015      1.75         1        1
 2 Klebsiella pn… Ertapenem       7 2015-01    2015      4.08         7        1
 3 Klebsiella pn… Ertapenem      13 2015-01    2015     10           13        1
 4 Klebsiella pn… Ertapenem       1 2015-02    2015      1.98         1        2
 5 Klebsiella pn… Ertapenem       7 2015-02    2015      4.44         7        2
 6 Klebsiella pn… Ertapenem      13 2015-02    2015      4.17        13        2
 7 Klebsiella pn… Ertapenem       1 2015-03    2015      0            1        3
 8 Klebsiella pn… Ertapenem       7 2015-03    2015      2.22         7        3
 9 Klebsiella pn… Ertapenem      13 2015-03    2015      2.94        13        3
10 Klebsiella pn… Ertapenem       1 2015-04    2015      3.2          1        4
# ℹ 1,

In [ ]:
%%R
unique(data$month_id)

  [1]   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
 [19]  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
 [37]  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
 [55]  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
 [73]  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
 [91]  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108


In [ ]:
%%R
library(sf)
library(spdep)

# โหลด shapefile ของจังหวัด
shp <- st_read("/content/gdrive/MyDrive/Senior Project/Spatial/Regions_no_province_boundaries.json")  # สมมุติว่าคุณมี shapefile แล้ว

# สร้าง adjacency matrix
nb <- poly2nb(shp)
nb2INLA("adjacency.graph", nb)

Reading layer `Regions_no_province_boundaries' from data source 
  `/content/gdrive/.shortcut-targets-by-id/1h5EYQx4w-VInRE_mIX08fjO5fl1RpQIz/Senior Project/Spatial/Regions_no_province_boundaries.json' 
  using driver `GeoJSON'
Simple feature collection with 13 features and 10 fields
Geometry type: MULTIPOLYGON
Dimension:     XY
Bounding box:  xmin: 97.34336 ymin: 5.613038 xmax: 105.637 ymax: 20.46503
Geodetic CRS:  WGS 84


In [ ]:
%%R
shp

Simple feature collection with 13 features and 10 fields
Geometry type: MULTIPOLYGON
Dimension:     XY
Bounding box:  xmin: 97.34336 ymin: 5.613038 xmax: 105.637 ymax: 20.46503
Geodetic CRS:  WGS 84
First 10 features:
   HealthRegion   id           ADM1_EN    ADM1_TH  ADM0_EN   ADM0_TH ADM0_PCODE
1             1 <NA>        Chiang Mai    เชียงใหม่ Thailand ประเทศไทย         TH
2             2 <NA>         Uttaradit     อุตรดิตถ์ Thailand ประเทศไทย         TH
3             3 <NA>          Chai Nat      ชัยนาท Thailand ประเทศไทย         TH
4             4 <NA>        Nonthaburi      นนทบุรี Thailand ประเทศไทย         TH
5             5 <NA>        Ratchaburi      ราชบุรี Thailand ประเทศไทย         TH
6             6 <NA>      Samut Prakan สมุทรปราการ Thailand ประเทศไทย         TH
7             7 <NA>         Khon Kaen     ขอนแก่น Thailand ประเทศไทย         TH
8             8 <NA>         Bueng Kan      บึงกาฬ Thailand ประเทศไทย         TH
9             9 <NA> Nakhon Ratchasima  นครราชสีม

In [ ]:
%%R
# แมปข้อมูล %R ของแต่ละจังหวัดใส่ shapefile
shp$HealthRegion_id	 <- as.numeric(as.factor(shp$HealthRegion))
data$Region_id <- match(data$REGION, shp$HealthRegion)

In [ ]:
%%R
data

# A tibble: 1,042 × 8
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ertapenem       1 2015-01    2015      1.75         1        1
 2 Klebsiella pn… Ertapenem       7 2015-01    2015      4.08         7        1
 3 Klebsiella pn… Ertapenem      13 2015-01    2015     10           13        1
 4 Klebsiella pn… Ertapenem       1 2015-02    2015      1.98         1        2
 5 Klebsiella pn… Ertapenem       7 2015-02    2015      4.44         7        2
 6 Klebsiella pn… Ertapenem      13 2015-02    2015      4.17        13        2
 7 Klebsiella pn… Ertapenem       1 2015-03    2015      0            1        3
 8 Klebsiella pn… Ertapenem       7 2015-03    2015      2.22         7        3
 9 Klebsiella pn… Ertapenem      13 2015-03    2015      2.94        13        3
10 Klebsiella pn… Ertapenem       1 2015-04    2015      3.2          1        4
# ℹ 1,

In [ ]:
%%R
unique(data$Region_id)

 [1]  1  7 13  4  6  5 10 11  2  9  3  8 12


In [ ]:
%%R
library(lubridate)

# ดึงเดือนออกมาจาก SPEC_DATE
data$month_num <- month(as.Date(paste0(data$SPEC_DATE, "-01")))

# สร้างตัวแปร sine/cosine สำหรับ seasonal effect
data$sin_month <- sin(2 * pi * data$month_num / 12)
data$cos_month <- cos(2 * pi * data$month_num / 12)


In [ ]:
%%R
library(lubridate)

# ดึงเดือนจาก SPEC_DATE
data$month <- month(as.Date(paste0(data$SPEC_DATE, "-01")))

# แบ่งฤดูกาลแบบไทย (ตัวอย่าง: 3 ฤดู)
data$season <- case_when(
  data$month %in% c(3, 4, 5) ~ "summer",   # มีนาคม–พฤษภาคม
  data$month %in% c(6, 7, 8, 9) ~ "rainy",  # มิถุนายน–กันยายน
  data$month %in% c(10, 11, 12, 1, 2) ~ "winter"  # ต.ค.–ก.พ.
)

# แปลงเป็นรหัสตัวเลข (ID)
data$season_id <- as.numeric(as.factor(data$season))


In [ ]:
%%R
data$region_year <- interaction(data$Region_id, data$Year)

In [ ]:
%%R
data$R_scaled <- (data$R_percent + 0.001) / 100

In [ ]:
%%R
data$R_scaled <- pmin(pmax(data$R_scaled, 0.0001), 0.9999)  # ตัดค่าขอบ

In [ ]:
%%R
data$R_logit <- log(data$R_scaled / (1 - data$R_scaled))  # หรือ: logit(x)

In [ ]:
%%R
# แบ่งชุดข้อมูล
data$set <- ifelse(data$Year %in% 2022:2023, "test", "train")
data

# A tibble: 1,042 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ertapenem       1 2015-01    2015      1.75         1        1
 2 Klebsiella pn… Ertapenem       7 2015-01    2015      4.08         7        1
 3 Klebsiella pn… Ertapenem      13 2015-01    2015     10           13        1
 4 Klebsiella pn… Ertapenem       1 2015-02    2015      1.98         1        2
 5 Klebsiella pn… Ertapenem       7 2015-02    2015      4.44         7        2
 6 Klebsiella pn… Ertapenem      13 2015-02    2015      4.17        13        2
 7 Klebsiella pn… Ertapenem       1 2015-03    2015      0            1        3
 8 Klebsiella pn… Ertapenem       7 2015-03    2015      2.22         7        3
 9 Klebsiella pn… Ertapenem      13 2015-03    2015      2.94        13        3
10 Klebsiella pn… Ertapenem       1 2015-04    2015      3.2          1        4
# ℹ 1

In [ ]:
%%R
train_data <- data[data$set == "train", ]
test_data <- data[data$set == "test", ]
train_data

# A tibble: 730 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ertapenem       1 2015-01    2015      1.75         1        1
 2 Klebsiella pn… Ertapenem       7 2015-01    2015      4.08         7        1
 3 Klebsiella pn… Ertapenem      13 2015-01    2015     10           13        1
 4 Klebsiella pn… Ertapenem       1 2015-02    2015      1.98         1        2
 5 Klebsiella pn… Ertapenem       7 2015-02    2015      4.44         7        2
 6 Klebsiella pn… Ertapenem      13 2015-02    2015      4.17        13        2
 7 Klebsiella pn… Ertapenem       1 2015-03    2015      0            1        3
 8 Klebsiella pn… Ertapenem       7 2015-03    2015      2.22         7        3
 9 Klebsiella pn… Ertapenem      13 2015-03    2015      2.94        13        3
10 Klebsiella pn… Ertapenem       1 2015-04    2015      3.2          1        4
# ℹ 720

In [ ]:
%%R
test_data$R_logit_real <- test_data$R_logit# เก็บของจริงไว้ใช้เทียบภายหลัง
test_data$R_logit <- NA  # ให้โมเดลพยากรณ์
test_data$R_logit_real

  [1] -0.9330015 -2.4078136 -2.0548512 -0.7016858 -1.6054368 -1.2416556
  [7] -0.6366617 -1.6620070 -1.1371552 -1.0741681 -2.4068132 -0.8369939
 [13] -1.3917718 -1.1493299 -2.9309857 -2.4625946 -0.9078737 -1.7530652
 [19] -1.5287891 -0.8254514 -1.7022114 -0.9797784 -1.1166235 -2.5175517
 [25] -0.8956229 -1.4375232 -1.2621835 -1.9195032 -2.4422112 -0.8543676
 [31] -1.9497789 -1.4637871 -1.1895282 -1.6093659 -0.8585104 -1.2473202
 [37] -2.5335500 -0.9162417 -2.0648133 -1.6555048 -1.9693476 -2.1221568
 [43] -0.2193223 -1.8994677 -1.6899007 -0.8223118 -1.5140601 -0.9768650
 [49] -0.9615559 -2.4102242 -1.0635913 -1.9973729 -1.5754660 -2.2405946
 [55] -2.7173569 -0.1027694 -2.1483275 -2.0328239 -0.8786515 -2.3615470
 [61] -0.7042753 -0.9243015 -2.5913621 -1.0915169 -1.9770691 -1.5162797
 [67] -2.2109056 -2.2511758 -0.4735606 -2.1165163 -1.3181809 -0.7824806
 [73] -1.7871634 -1.0764592 -1.3665081 -2.1548945 -1.5443365 -1.7652926
 [79] -1.3961818 -2.1632149 -2.9242980 -1.3942637 -2.9309857 -1.

In [ ]:
%%R
# Add the R_scaled_real column to train_data and fill it with NAs
train_data$R_logit_real <- NA

# Now, combine the data frames
combined_data <- rbind(train_data, test_data)
combined_data

# A tibble: 1,042 × 19
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ertapenem       1 2015-01    2015      1.75         1        1
 2 Klebsiella pn… Ertapenem       7 2015-01    2015      4.08         7        1
 3 Klebsiella pn… Ertapenem      13 2015-01    2015     10           13        1
 4 Klebsiella pn… Ertapenem       1 2015-02    2015      1.98         1        2
 5 Klebsiella pn… Ertapenem       7 2015-02    2015      4.44         7        2
 6 Klebsiella pn… Ertapenem      13 2015-02    2015      4.17        13        2
 7 Klebsiella pn… Ertapenem       1 2015-03    2015      0            1        3
 8 Klebsiella pn… Ertapenem       7 2015-03    2015      2.22         7        3
 9 Klebsiella pn… Ertapenem      13 2015-03    2015      2.94        13        3
10 Klebsiella pn… Ertapenem       1 2015-04    2015      3.2          1        4
# ℹ 1

In [ ]:
%%R

formula <-  R_percent ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1")+ sin_month + cos_month + f(region_year, model="iid")
#formula <- R_percent ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1")
result <- inla(formula,
               family="gaussian",
               data=combined_data,
               control.predictor=list(compute=TRUE),
               control.compute=list(dic=TRUE),
               control.inla = list(strategy = "simplified.laplace"))

# ดูผลลัพธ์
summary(result)

Time used:
    Pre = 15.9, Running = 2.05, Post = 0.0728, Total = 18 
Fixed effects:
              mean    sd 0.025quant 0.5quant 0.975quant   mode kld
(Intercept) 11.997 0.698     10.611   11.999     13.371 11.998   0
sin_month    0.244 0.212     -0.174    0.244      0.658  0.244   0
cos_month    0.756 0.199      0.366    0.756      1.145  0.756   0

Random effects:
  Name	  Model
    Region_id BYM2 model
   month_id RW2 model
   Year RW1 model
   region_year IID model

Model hyperparameters:
                                            mean       sd 0.025quant 0.5quant
Precision for the Gaussian observations 5.40e-02 3.00e-03      0.049 5.40e-02
Precision for Region_id                 3.18e-01 1.89e-01      0.108 2.70e-01
Phi for Region_id                       5.59e-01 2.94e-01      0.062 5.81e-01
Precision for month_id                  2.05e+02 1.81e+02     35.479 1.53e+02
Precision for Year                      2.22e+04 2.70e+04   1033.278 1.34e+04
Precision for region_year        

In [ ]:
%%R
combined_data$predicted_percent <- result$summary.fitted.values$mean

In [ ]:
%%R
test_data$predicted_percent <- combined_data$predicted_percent[combined_data$set == "test"]

In [ ]:
%%R
test_data$actual_percent <- test_data$R_percent

In [ ]:
%%R
wape <- sum(abs(test_data$predicted_percent - test_data$actual_percent)) /
        sum(abs(test_data$actual_percent))

wape_percent <- round(wape * 100, 2)

cat("📊 WAPE for 2022–2023 =", wape_percent, "%\n")

📊 WAPE for 2022–2023 = 15.15 %


In [ ]:
%%R
test_data_valid <- test_data[test_data$actual_percent > 0, ]

mape_val <- mean(abs(test_data_valid$predicted_percent - test_data_valid$actual_percent) /
                 test_data_valid$actual_percent) * 100

cat("📈 MAPE =", round(mape_val, 2), "%\n")

📈 MAPE = 19.73 %


##Predict

In [ ]:
%%R
data

# A tibble: 1,042 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ertapenem       1 2015-01    2015      1.75         1        1
 2 Klebsiella pn… Ertapenem       7 2015-01    2015      4.08         7        1
 3 Klebsiella pn… Ertapenem      13 2015-01    2015     10           13        1
 4 Klebsiella pn… Ertapenem       1 2015-02    2015      1.98         1        2
 5 Klebsiella pn… Ertapenem       7 2015-02    2015      4.44         7        2
 6 Klebsiella pn… Ertapenem      13 2015-02    2015      4.17        13        2
 7 Klebsiella pn… Ertapenem       1 2015-03    2015      0            1        3
 8 Klebsiella pn… Ertapenem       7 2015-03    2015      2.22         7        3
 9 Klebsiella pn… Ertapenem      13 2015-03    2015      2.94        13        3
10 Klebsiella pn… Ertapenem       1 2015-04    2015      3.2          1        4
# ℹ 1

In [ ]:
%%R
data

# A tibble: 1,042 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ertapenem       1 2015-01    2015      1.75         1        1
 2 Klebsiella pn… Ertapenem       7 2015-01    2015      4.08         7        1
 3 Klebsiella pn… Ertapenem      13 2015-01    2015     10           13        1
 4 Klebsiella pn… Ertapenem       1 2015-02    2015      1.98         1        2
 5 Klebsiella pn… Ertapenem       7 2015-02    2015      4.44         7        2
 6 Klebsiella pn… Ertapenem      13 2015-02    2015      4.17        13        2
 7 Klebsiella pn… Ertapenem       1 2015-03    2015      0            1        3
 8 Klebsiella pn… Ertapenem       7 2015-03    2015      2.22         7        3
 9 Klebsiella pn… Ertapenem      13 2015-03    2015      2.94        13        3
10 Klebsiella pn… Ertapenem       1 2015-04    2015      3.2          1        4
# ℹ 1

In [ ]:
%%R
future_years <- 2024:2028
future_months <- 1:12
future_regions <- sort(unique(data$REGION))  # หรือ Health Region ที่มี
future_GROUP_NAME <- unique(data$GROUP_NAME)
future_ANTIBIOTIC <- unique(data$ANTIBIOTIC)
# สร้าง grid ของข้อมูลล่วงหน้า
future_data <- expand.grid(
  GROUP_NAME = future_GROUP_NAME,
  ANTIBIOTIC = future_ANTIBIOTIC,
  REGION = future_regions,
  month = future_months,
  Year = future_years
)
future_data

               GROUP_NAME ANTIBIOTIC REGION month Year
1   Klebsiella pneumoniae  Ertapenem      1     1 2024
2   Klebsiella pneumoniae  Ertapenem      2     1 2024
3   Klebsiella pneumoniae  Ertapenem      3     1 2024
4   Klebsiella pneumoniae  Ertapenem      4     1 2024
5   Klebsiella pneumoniae  Ertapenem      5     1 2024
6   Klebsiella pneumoniae  Ertapenem      6     1 2024
7   Klebsiella pneumoniae  Ertapenem      7     1 2024
8   Klebsiella pneumoniae  Ertapenem      8     1 2024
9   Klebsiella pneumoniae  Ertapenem      9     1 2024
10  Klebsiella pneumoniae  Ertapenem     10     1 2024
11  Klebsiella pneumoniae  Ertapenem     11     1 2024
12  Klebsiella pneumoniae  Ertapenem     12     1 2024
13  Klebsiella pneumoniae  Ertapenem     13     1 2024
14  Klebsiella pneumoniae  Ertapenem      1     2 2024
15  Klebsiella pneumoniae  Ertapenem      2     2 2024
16  Klebsiella pneumoniae  Ertapenem      3     2 2024
17  Klebsiella pneumoniae  Ertapenem      4     2 2024
18  Klebsi

In [ ]:
%%R
library(dplyr)

future_data <- future_data %>%
  mutate(SPEC_DATE = sprintf("%d-%02d", Year, month))
future_data

               GROUP_NAME ANTIBIOTIC REGION month Year SPEC_DATE
1   Klebsiella pneumoniae  Ertapenem      1     1 2024   2024-01
2   Klebsiella pneumoniae  Ertapenem      2     1 2024   2024-01
3   Klebsiella pneumoniae  Ertapenem      3     1 2024   2024-01
4   Klebsiella pneumoniae  Ertapenem      4     1 2024   2024-01
5   Klebsiella pneumoniae  Ertapenem      5     1 2024   2024-01
6   Klebsiella pneumoniae  Ertapenem      6     1 2024   2024-01
7   Klebsiella pneumoniae  Ertapenem      7     1 2024   2024-01
8   Klebsiella pneumoniae  Ertapenem      8     1 2024   2024-01
9   Klebsiella pneumoniae  Ertapenem      9     1 2024   2024-01
10  Klebsiella pneumoniae  Ertapenem     10     1 2024   2024-01
11  Klebsiella pneumoniae  Ertapenem     11     1 2024   2024-01
12  Klebsiella pneumoniae  Ertapenem     12     1 2024   2024-01
13  Klebsiella pneumoniae  Ertapenem     13     1 2024   2024-01
14  Klebsiella pneumoniae  Ertapenem      1     2 2024   2024-02
15  Klebsiella pneumoniae

In [ ]:
%%R
# เข้ารหัสพื้นที่ (province) และเวลา (month)
future_data$Region_id <- as.numeric(as.factor(future_data$REGION))
# สร้าง factor แล้วแปลงเป็นเลขที่เรียง
future_data$month_id <- as.numeric(as.factor(future_data$SPEC_DATE))

# สร้าง region-year interaction (ถ้าใช้ในโมเดล)
future_data$region_year <- interaction(future_data$Region_id, future_data$Year)

# เติมค่าที่ยังไม่มี (target)
future_data$R_logit <- NA  # หรือ R_scaled หรือ R_percent ตามที่ใช้
future_data$R_scaled <- NA
future_data$R_percent<- NA
head(future_data,10)

              GROUP_NAME ANTIBIOTIC REGION month Year SPEC_DATE Region_id
1  Klebsiella pneumoniae  Ertapenem      1     1 2024   2024-01         1
2  Klebsiella pneumoniae  Ertapenem      2     1 2024   2024-01         2
3  Klebsiella pneumoniae  Ertapenem      3     1 2024   2024-01         3
4  Klebsiella pneumoniae  Ertapenem      4     1 2024   2024-01         4
5  Klebsiella pneumoniae  Ertapenem      5     1 2024   2024-01         5
6  Klebsiella pneumoniae  Ertapenem      6     1 2024   2024-01         6
7  Klebsiella pneumoniae  Ertapenem      7     1 2024   2024-01         7
8  Klebsiella pneumoniae  Ertapenem      8     1 2024   2024-01         8
9  Klebsiella pneumoniae  Ertapenem      9     1 2024   2024-01         9
10 Klebsiella pneumoniae  Ertapenem     10     1 2024   2024-01        10
   month_id region_year R_logit R_scaled R_percent
1         1      1.2024      NA       NA        NA
2         1      2.2024      NA       NA        NA
3         1      3.2024      NA  

In [ ]:
%%R
library(lubridate)

# ดึงเดือนออกมาจาก SPEC_DATE
future_data$month_num <- month(as.Date(paste0(future_data$SPEC_DATE, "-01")))

# สร้างตัวแปร sine/cosine สำหรับ seasonal effect
future_data$sin_month <- sin(2 * pi * future_data$month_num / 12)
future_data$cos_month <- cos(2 * pi * future_data$month_num / 12)
head(future_data,10)

              GROUP_NAME ANTIBIOTIC REGION month Year SPEC_DATE Region_id
1  Klebsiella pneumoniae  Ertapenem      1     1 2024   2024-01         1
2  Klebsiella pneumoniae  Ertapenem      2     1 2024   2024-01         2
3  Klebsiella pneumoniae  Ertapenem      3     1 2024   2024-01         3
4  Klebsiella pneumoniae  Ertapenem      4     1 2024   2024-01         4
5  Klebsiella pneumoniae  Ertapenem      5     1 2024   2024-01         5
6  Klebsiella pneumoniae  Ertapenem      6     1 2024   2024-01         6
7  Klebsiella pneumoniae  Ertapenem      7     1 2024   2024-01         7
8  Klebsiella pneumoniae  Ertapenem      8     1 2024   2024-01         8
9  Klebsiella pneumoniae  Ertapenem      9     1 2024   2024-01         9
10 Klebsiella pneumoniae  Ertapenem     10     1 2024   2024-01        10
   month_id region_year R_logit R_scaled R_percent month_num sin_month
1         1      1.2024      NA       NA        NA         1       0.5
2         1      2.2024      NA       NA    

In [ ]:
%%R
library(lubridate)

# ดึงเดือนจาก SPEC_DATE
future_data$month <- month(as.Date(paste0(future_data$SPEC_DATE, "-01")))

# แบ่งฤดูกาลแบบไทย (ตัวอย่าง: 3 ฤดู)
future_data$season <- case_when(
  future_data$month %in% c(3, 4, 5) ~ "summer",   # มีนาคม–พฤษภาคม
  future_data$month %in% c(6, 7, 8, 9) ~ "rainy",  # มิถุนายน–กันยายน
  future_data$month %in% c(10, 11, 12, 1, 2) ~ "winter"  # ต.ค.–ก.พ.
)

# แปลงเป็นรหัสตัวเลข (ID)
future_data$season_id <- as.numeric(as.factor(future_data$season))
head(future_data,10)

              GROUP_NAME ANTIBIOTIC REGION month Year SPEC_DATE Region_id
1  Klebsiella pneumoniae  Ertapenem      1     1 2024   2024-01         1
2  Klebsiella pneumoniae  Ertapenem      2     1 2024   2024-01         2
3  Klebsiella pneumoniae  Ertapenem      3     1 2024   2024-01         3
4  Klebsiella pneumoniae  Ertapenem      4     1 2024   2024-01         4
5  Klebsiella pneumoniae  Ertapenem      5     1 2024   2024-01         5
6  Klebsiella pneumoniae  Ertapenem      6     1 2024   2024-01         6
7  Klebsiella pneumoniae  Ertapenem      7     1 2024   2024-01         7
8  Klebsiella pneumoniae  Ertapenem      8     1 2024   2024-01         8
9  Klebsiella pneumoniae  Ertapenem      9     1 2024   2024-01         9
10 Klebsiella pneumoniae  Ertapenem     10     1 2024   2024-01        10
   month_id region_year R_logit R_scaled R_percent month_num sin_month
1         1      1.2024      NA       NA        NA         1       0.5
2         1      2.2024      NA       NA    

In [ ]:
%%R
future_data$set <- NA
head(future_data,10)

              GROUP_NAME ANTIBIOTIC REGION month Year SPEC_DATE Region_id
1  Klebsiella pneumoniae  Ertapenem      1     1 2024   2024-01         1
2  Klebsiella pneumoniae  Ertapenem      2     1 2024   2024-01         2
3  Klebsiella pneumoniae  Ertapenem      3     1 2024   2024-01         3
4  Klebsiella pneumoniae  Ertapenem      4     1 2024   2024-01         4
5  Klebsiella pneumoniae  Ertapenem      5     1 2024   2024-01         5
6  Klebsiella pneumoniae  Ertapenem      6     1 2024   2024-01         6
7  Klebsiella pneumoniae  Ertapenem      7     1 2024   2024-01         7
8  Klebsiella pneumoniae  Ertapenem      8     1 2024   2024-01         8
9  Klebsiella pneumoniae  Ertapenem      9     1 2024   2024-01         9
10 Klebsiella pneumoniae  Ertapenem     10     1 2024   2024-01        10
   month_id region_year R_logit R_scaled R_percent month_num sin_month
1         1      1.2024      NA       NA        NA         1       0.5
2         1      2.2024      NA       NA    

In [ ]:
%%R
head(data,10)

# A tibble: 10 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ertapenem       1 2015-01    2015      1.75         1        1
 2 Klebsiella pn… Ertapenem       7 2015-01    2015      4.08         7        1
 3 Klebsiella pn… Ertapenem      13 2015-01    2015     10           13        1
 4 Klebsiella pn… Ertapenem       1 2015-02    2015      1.98         1        2
 5 Klebsiella pn… Ertapenem       7 2015-02    2015      4.44         7        2
 6 Klebsiella pn… Ertapenem      13 2015-02    2015      4.17        13        2
 7 Klebsiella pn… Ertapenem       1 2015-03    2015      0            1        3
 8 Klebsiella pn… Ertapenem       7 2015-03    2015      2.22         7        3
 9 Klebsiella pn… Ertapenem      13 2015-03    2015      2.94        13        3
10 Klebsiella pn… Ertapenem       1 2015-04    2015      3.2          1        4
# ℹ 10 m

In [ ]:
%%R
combined_data_future <- rbind(data, future_data)
combined_data_future

# A tibble: 1,822 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <dbl>    <dbl>
 1 Klebsiella pn… Ertapenem       1 2015-01    2015      1.75         1        1
 2 Klebsiella pn… Ertapenem       7 2015-01    2015      4.08         7        1
 3 Klebsiella pn… Ertapenem      13 2015-01    2015     10           13        1
 4 Klebsiella pn… Ertapenem       1 2015-02    2015      1.98         1        2
 5 Klebsiella pn… Ertapenem       7 2015-02    2015      4.44         7        2
 6 Klebsiella pn… Ertapenem      13 2015-02    2015      4.17        13        2
 7 Klebsiella pn… Ertapenem       1 2015-03    2015      0            1        3
 8 Klebsiella pn… Ertapenem       7 2015-03    2015      2.22         7        3
 9 Klebsiella pn… Ertapenem      13 2015-03    2015      2.94        13        3
10 Klebsiella pn… Ertapenem       1 2015-04    2015      3.2          1        4
# ℹ 1

In [ ]:
%%R
# จากนั้นใช้ formula แบบ besag ได้เลย
formula <-  R_percent ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1")+ sin_month + cos_month + f(region_year, model="iid")
#formula <- R_percent ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1")
result <- inla(formula,
               family="gaussian",
               data=combined_data_future,
               control.predictor=list(compute=TRUE),
               control.compute=list(dic=TRUE),
               control.inla = list(strategy = "simplified.laplace"))

# ดูผลลัพธ์
summary(result)

Time used:
    Pre = 24.4, Running = 4.2, Post = 0.0723, Total = 28.7 
Fixed effects:
              mean    sd 0.025quant 0.5quant 0.975quant   mode kld
(Intercept) 11.998 0.682     10.647   11.999     13.339 11.999   0
sin_month    0.243 0.212     -0.174    0.243      0.658  0.243   0
cos_month    0.756 0.199      0.366    0.756      1.145  0.756   0

Random effects:
  Name	  Model
    Region_id BYM2 model
   month_id RW2 model
   Year RW1 model
   region_year IID model

Model hyperparameters:
                                            mean       sd 0.025quant 0.5quant
Precision for the Gaussian observations 5.40e-02 2.00e-03      0.049 5.40e-02
Precision for Region_id                 3.38e-01 2.06e-01      0.106 2.87e-01
Phi for Region_id                       5.65e-01 2.54e-01      0.092 5.86e-01
Precision for month_id                  2.03e+02 1.80e+02     37.858 1.52e+02
Precision for Year                      2.19e+04 2.42e+04   1377.690 1.43e+04
Precision for region_year       

In [ ]:
%%R
# คำนวณจำนวนแถว
n_total       <- nrow(combined_data_future)
n_future      <- nrow(future_data)
start_index   <- n_total - n_future + 1
end_index     <- n_total

# ดึงค่าที่พยากรณ์เฉพาะ future
future_data$predicted_percent <- result$summary.fitted.values$mean[start_index:end_index]

In [ ]:
%%R
head(future_data[, c("REGION", "SPEC_DATE", "predicted_percent")],13)

   REGION SPEC_DATE predicted_percent
1       1   2024-01        3.21304709
2       2   2024-01        2.56335814
3       3   2024-01       -1.87030486
4       4   2024-01        3.12208054
5       5   2024-01        0.02028202
6       6   2024-01        4.41029597
7       7   2024-01        5.49643316
8       8   2024-01        4.72151210
9       9   2024-01        3.53846276
10     10   2024-01        3.00807500
11     11   2024-01       -2.32973793
12     12   2024-01       -0.65144199
13     13   2024-01        2.04052349


In [ ]:
%%R
install.packages("writexl")  # รันครั้งเดียว
library(writexl)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/writexl_1.5.4.tar.gz'
Content type 'application/x-gzip' length 278312 bytes (271 KB)
downloaded 271 KB


The downloaded source packages are in
	‘/tmp/RtmpBFeRgL/downloaded_packages’


In [ ]:
%%R
write_xlsx(future_data, path = "/content/gdrive/MyDrive/Senior Project/Spatial/Spatiotemporal/results/Klebsiella pneumoniae/Ertapenem/Future_R_Klebsiella pneumoniae_Ertapenem.xlsx")